In [ ]:
# ── Connection — Variable Library VL_SALES_ORD ────────────────────────────────
# SOURCE_LH_ABFSS in VL ends with /Tables. SD transactional tables are under
# Tables/dbo/{name}, so SOURCE_SCHEMA = 'dbo'.
# SD desc tables override this with source_schema='' (flat path in MD source).
_vl = notebookutils.variableLibrary.getLibrary('VL_SALES_ORD')

SOURCE_LH_ABFSS  = _vl['SOURCE_LH_ABFSS'].strip()
SOURCE_SCHEMA    = 'dbo'

BRONZE_LH_ABFSS  = _vl['BRONZE_LH_ABFSS'].strip()
BRONZE_SCHEMA    = 'dbo'

OPS_LH_ABFSS     = _vl['OPS_LH_ABFSS'].strip()
OPS_SCHEMA       = 'dbo'

PIPELINE_NAME    = 'SD_Bronze'
PIPELINE_RUN_ID  = ''   # blank = auto-generate

In [ ]:
%run ./NB_Config_SALES_ORD_v2

In [ ]:
%run ./NB_Utils_Bronze_v2

In [ ]:
# ── Pipeline run setup ────────────────────────────────────────────────────────
from datetime import datetime
import uuid

if not PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = str(uuid.uuid4())

start_time = datetime.utcnow()
print(f'Pipeline : {PIPELINE_NAME}')
print(f'Run ID   : {PIPELINE_RUN_ID}')
print(f'Started  : {start_time}')

In [ ]:
# ── Executor — group by source lakehouse, call run_pipeline per group ─────────
# SD_TABLE_CONFIGS: transactional tables, source_schema='' (VL path includes /Tables)
# SD_DESC_CONFIGS:  desc/lookup tables,   source_schema='' (flat, no dbo prefix)
DEFAULT_LH_KEY = 'SOURCE_LH_ABFSS'
all_configs = {**SD_TABLE_CONFIGS, **SD_DESC_CONFIGS}
groups = {}
for name, cfg in all_configs.items():
    lh_key = cfg.get('source_lh_key', DEFAULT_LH_KEY)
    groups.setdefault(lh_key, {})[name] = cfg

all_results, all_skipped = [], []
for lh_key, group_configs in groups.items():
    SOURCE_LH_ABFSS = _vl[lh_key].strip()
    r, s = run_pipeline(group_configs)
    all_results.extend(r)
    all_skipped.extend(s)

results, skipped = all_results, all_skipped

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print_summary(results, skipped, 'SD', start_time)